# Exploration du dataset
[Lien d'origine Databricks](https://dbc-cb7385f9-5ab2.cloud.databricks.com/editor/notebooks/3309334937974042?o=7474660403768866)

## Exploration & Qualité des Données (Recettes)
#### Ce notebook a pour but d'analyser la qualité, la complétude et la distribution des données générées par le pipeline de préparation (`recipes_main`, `ingredients_index`, `recipes_nutrition_detail`).

## 1. Configuration & Imports
#### Chargement des tables au format Delta. L'utilisation du format Delta nous assure des temps de lecture optimisés grâce au Data Skipping et au Z-Order calculés lors du pipeline.

In [1]:
try:
    spark.stop()
except Exception as e:
    pass

In [2]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Create a Spark session
spark = (SparkSession.builder
    .appName("RecipePipelineFinal")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.executor.memory", "3g")  # Mémoire allouée à chaque nœud de calcul
    .config("spark.driver.memory", "1g")    # Mémoire allouée au chef d'orchestre
    .getOrCreate()
)

In [3]:
# Définition du chemin
OUTPUT_PATH = "../data/recipes_parquets"

# Chargement des trois tables optimisées
df_main = spark.read.format("delta").load(f"{OUTPUT_PATH}/recipes_main")
df_index = spark.read.format("delta").load(f"{OUTPUT_PATH}/ingredients_index")
df_nutrition_detail = spark.read.format("delta").load(f"{OUTPUT_PATH}/recipes_nutrition_detail")

total_rows_main = df_main.count()

print(f"✅ Tables chargées avec succès.")
print(f"Nombre total de recettes uniques : {total_rows_main:,}")

26/04/01 09:43:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✅ Tables chargées avec succès.
Nombre total de recettes uniques : 1,029,720


## 2. Dictionnaire des Données (Data Dictionary)
#### **Table `recipes_main` (Table principale)**
* `recipe_id` : ID du dataset MIT
* `title` : Nom de la recette
* `description` : Commentaires sur la recette
* `instructions_text` : Étapes de préparation
* `ingredients_raw` : Liste brute des ingrédients avec quantités
* `ingredients_validated` : Array des ingrédients reconnus par le NLP
* `n_ingredients_validated` : Nombre d'ingrédients de l'array
* `n_steps` : Nombre d'étapes de préparation
* `cook_minutes` : Temps de préparation
* `cook_time_category` : Catégorie de la rapidité de préparation : rapide<=30 ~ 30<moyen<60 ~ 60<=long
* `image_url` : URL de l'image du plat
* `image_urls` : Liste de toutes les URLs d'image
* `has_image` : Vérifie si on a une image - Booléen
* `source_url` : Site internet de la recette
* `energy_kcal` : Nombre de kcal
* `nutri_score` : Score de A à E basé sur les kcal
* `tags` : Liste de tags pour faire une recherche rapide

#### **Table `ingredients_index` (Index pour la recherche)**
* `recipe_id` : ID du dataset MIT
* `title` : Nom de la recette
* `nutri_score` : Score de A à E basé sur les kcal
* `image_url` : URL de l'image du plat
* `cook_time_category` : Catégorie de la rapidité de préparation
* `ingredient` : Ingrédient unique éclaté pour la recherche inversée

#### **Table `recipes_nutrition_detail` (Macros)**
* `fat_g` / `protein_g` / `salt_g` / `sugars_g` : Macronutriments en grammes (float)

In [4]:
# Remplacement de display() par .toPandas() pour conserver un affichage propre dans Jupyter
display(df_main.limit(2).toPandas())
display(df_index.limit(2).toPandas())
display(df_nutrition_detail.limit(2).toPandas())

,recipe_id,title,description,instructions_text,ingredients_raw,ingredients_validated,n_ingredients_validated,n_steps,cook_minutes,cook_time_category,image_url,image_urls,has_image,source_url,energy_kcal,nutri_score,tags
0,33b10f84ac,Pumpkin Creme Caramel,this is a delightful autumn dessert that is re...,Preheat oven to 350F. | Lightly coat 8 6-oz. |...,"[23 cup plus 1/2 cup sugar, divided, 1/2 tsp. ...","[sugar, salt, eggs, pumpkin puree, vanilla ext...",8,14,65,long,http://recipegreat.com/images/pumpkin-creme-ca...,[http://recipegreat.com/images/pumpkin-creme-c...,True,http://www.vegetariantimes.com/recipe/pumpkin-...,247.100006,C,"[time-to-make, course, main-ingredient, prepar..."
1,0e38daa3a7,Chocolate Cake Donuts,this recipe came on the packaging of my new do...,"For the donuts, mix dry ingredients in bowl. |...","[2 cups Flour, 3/4 cups Sugar, 1/2 cups Unswee...","[flour, sugar, unsweetened cocoa powder, bakin...",18,13,26,rapide,http://tastykitchen.com/recipes/wp-content/upl...,[http://tastykitchen.com/recipes/wp-content/up...,True,http://tastykitchen.com/recipes/breakfastbrunc...,181.199997,C,"[30-minutes-or-less, time-to-make, course, pre..."


,recipe_id,title,nutri_score,image_url,cook_time_category,ingredient
0,07507ea3fa,Breakfast Scones,A,https://s-media-cache-ak0.pinimg.com/originals...,rapide,flour
1,07507ea3fa,Breakfast Scones,A,https://s-media-cache-ak0.pinimg.com/originals...,rapide,eggs


,recipe_id,fat_g,protein_g,salt_g,saturates_g,sugars_g
0,00adf8c120,4.391415,11.111555,0.904074,1.734996,2.864545
1,012cbe55b8,5.469795,3.140385,0.094987,3.418553,4.918221


## 3. Échantillonnage : Réalité vs Idéal
#### Confrontation entre un échantillon aléatoire (qui représente la réalité du dataset avec ses données potentiellement manquantes) et un échantillon "Parfait" où toutes les sources (Kaggle, MIT, Nutrition) ont pu être croisées avec succès.

In [5]:
print("--- Option 1 : 5 recettes prises au hasard ---")
display(df_main.limit(5).toPandas())

--- Option 1 : 5 recettes prises au hasard ---


,recipe_id,title,description,instructions_text,ingredients_raw,ingredients_validated,n_ingredients_validated,n_steps,cook_minutes,cook_time_category,image_url,image_urls,has_image,source_url,energy_kcal,nutri_score,tags
0,33b10f84ac,Pumpkin Creme Caramel,this is a delightful autumn dessert that is re...,Preheat oven to 350F. | Lightly coat 8 6-oz. |...,"[23 cup plus 1/2 cup sugar, divided, 1/2 tsp. ...","[sugar, salt, eggs, pumpkin puree, vanilla ext...",8,14.0,65.0,long,http://recipegreat.com/images/pumpkin-creme-ca...,[http://recipegreat.com/images/pumpkin-creme-c...,True,http://www.vegetariantimes.com/recipe/pumpkin-...,247.100006,C,"[time-to-make, course, main-ingredient, prepar..."
1,0e38daa3a7,Chocolate Cake Donuts,this recipe came on the packaging of my new do...,"For the donuts, mix dry ingredients in bowl. |...","[2 cups Flour, 3/4 cups Sugar, 1/2 cups Unswee...","[flour, sugar, unsweetened cocoa powder, bakin...",18,13.0,26.0,rapide,http://tastykitchen.com/recipes/wp-content/upl...,[http://tastykitchen.com/recipes/wp-content/up...,True,http://tastykitchen.com/recipes/breakfastbrunc...,181.199997,C,"[30-minutes-or-less, time-to-make, course, pre..."
2,e8410d37e8,Knock You Nakeds,"a sweet, gooey, chocolatey type of brownie bar...",Preheat oven to 180c& grease and line a 8 x 13...,"[1 box chocolate cake mix, 1 cup chopped nuts,...","[nuts, evaporated milk, butter, caramel candie...",6,8.0,50.0,moyen,http://4.bp.blogspot.com/-ioJSCd9heHY/UW6Rj014...,[http://4.bp.blogspot.com/-ioJSCd9heHY/UW6Rj01...,True,http://www.food.com/recipe/knock-you-nakeds-66323,255.899994,C,"[60-minutes-or-less, time-to-make, course, mai..."
3,68b2035a2c,Mini Parmesan Scones,NaN,Preheat oven to 350 degrees F (175 degrees C)....,"[2 tablespoons butter, 2 cups self-rising flou...","[butter, self - rising flour, salt, parmesan c...",7,NaN,NaN,inconnu,http://images.media-allrecipes.com/userphotos/...,[http://images.media-allrecipes.com/userphotos...,True,http://allrecipes.com/recipe/mini-parmesan-sco...,241.493927,C,None
4,5a616ae73b,Italian Wine Biscuits,another traditional italian cookie well known ...,Preheat oven to 375. | Mix wet ingredients int...,"[4 cups all-purpose flour, 1 cup sugar, 1 cup ...","[all - purpose flour, sugar, red table wine, b...",5,7.0,30.0,rapide,http://4.bp.blogspot.com/-qZPmoj3djvY/UIFVZP8t...,[http://4.bp.blogspot.com/-qZPmoj3djvY/UIFVZP8...,True,http://www.food.com/recipe/italian-wine-biscui...,177.000000,C,"[30-minutes-or-less, time-to-make, course, cui..."


In [6]:
print("--- Option 2 : 5 recettes 'Parfaites' ---")
# On force le filtre pour voir à quoi ressemble une recette où TOUTES les sources ont matché
df_perfect = df_main.filter(
    (F.col("has_image") == True) & 
    (F.col("energy_kcal").isNotNull()) & 
    (F.col("n_ingredients_validated") > 0)
)

display(df_perfect.limit(5).toPandas())

--- Option 2 : 5 recettes 'Parfaites' ---


,recipe_id,title,description,instructions_text,ingredients_raw,ingredients_validated,n_ingredients_validated,n_steps,cook_minutes,cook_time_category,image_url,image_urls,has_image,source_url,energy_kcal,nutri_score,tags
0,ac1e840ffc,Couscous Paella with Shrimp,yummy dish with lots of ingredients i love.,Heat oil in a 8 to 10 quart heavy bottomed pot...,"[1 tablespoon olive oil, 1 medium onion, chopp...","[olive oil, onion, garlic cloves, sweet red pe...",16,11.0,35.0,moyen,https://static01.nyt.com/images/2014/06/25/din...,[https://static01.nyt.com/images/2014/06/25/di...,True,http://www.foodnetwork.com/recipes/couscous-pa...,402.600006,E,"[60-minutes-or-less, time-to-make, course, pre..."
1,2f08b721c7,Grilled Mediterranean Kebabs on Pita,"it was a nice warm day, and i had some lamb an...","If using bamboo skewers, soak them submerged i...",[1 12 lbs ground lamb (may use up to 50% groun...,"[ground lamb, eggs, white onion, garlic cloves...",21,8.0,72.0,long,"http://img.sndimg.com/food/image/upload/w_512,...",[http://img.sndimg.com/food/image/upload/w_512...,True,http://www.food.com/recipe/grilled-mediterrane...,944.099976,E,"[time-to-make, course, main-ingredient, cuisin..."
2,91ed9d3edd,Energy Bar (Clif / Larabar),NaN,"Blend dates, almonds, peanuts and cinnamon in ...","[1 12 cups dates, 12 cup almonds, 12 cup peanu...","[dates, almonds, peanuts, cinnamon, quick oats...",6,NaN,NaN,inconnu,http://s3.amazonaws.com/jo.www.larabar.com/upl...,[http://s3.amazonaws.com/jo.www.larabar.com/up...,True,http://www.food.com/recipe/energy-bar-clif-lar...,639.873962,E,None
3,8e18d50d79,Creamy Chicken Dip,this dip is creamy and delicious. easy to make...,"In a mixing bowl beat cream cheese, sour cream...","[8 ounces cream cheese, softened, 8 ounces sou...","[cream cheese, sour cream, Worcestershire sauc...",6,6.0,40.0,moyen,http://4.bp.blogspot.com/-kIKGgqAOAPA/Unu685Ns...,[http://4.bp.blogspot.com/-kIKGgqAOAPA/Unu685N...,True,http://www.food.com/recipe/creamy-chicken-dip-...,1622.000000,E,"[60-minutes-or-less, time-to-make, course, pre..."
4,3d84b1700d,Ham and Noodle Casserole,easy and delicious! great week night meal when...,Cook and drain egg noodles and combine with di...,"[8 ounces wide egg noodles, broken up, 2 cups ...","[wide egg noodles, ham, half - and - half, egg...",8,5.0,65.0,long,http://www.bloggingwithapples.com/wp-content/u...,[http://www.bloggingwithapples.com/wp-content/...,True,http://www.food.com/recipe/ham-and-noodle-cass...,542.599976,E,"[ham, time-to-make, course, main-ingredient, p..."


## 4. Analyse de Complétude (Data Quality)
#### Quelle est la proportion de notre catalogue qui est pleinement exploitable pour l'interface utilisateur (UI) ? Cette étape calcule globalement la complétude de notre base et détaille le pourcentage de valeurs non-nulles pour chaque colonne.

In [7]:
print("--- Option 3 : Comparaison des recettes 'Parfaites', 'Partielles' et 'Incomplètes' ---")

# 1a. Pour vérifier TOUTES les tables, on crée une vue globale (Left Join)
df_all_data = df_main.join(df_nutrition_detail, on="recipe_id", how="left")

# 1b. Nous appliquons un répartitionnement pour découper le dataset en plusieurs morceaux
df_all_data = df_all_data.repartition(200)

# 2. Condition Partielle : Ce qui est strictement nécessaire pour l'UI
condition_partielle = (
    (F.col("has_image") == True) &
    (F.col("energy_kcal").isNotNull()) &
    (F.col("n_ingredients_validated") > 0)
)

# 3. Condition Parfaite : Génération dynamique pour TOUTES les colonnes du DataFrame joint
condition_parfaite = F.lit(True)
for col_name in df_all_data.columns:
    condition_parfaite = condition_parfaite & F.col(col_name).isNotNull()

# 4. Ajout de la colonne de catégorisation et calcul des totaux
df_completude = (df_all_data
    .withColumn("statut_recette",
                F.when(condition_parfaite, F.lit("100% Parfaite (Toutes les colonnes des tables sont remplies)"))
                 .when(condition_partielle, F.lit("Partielle (Image + Nutrition de base + Ingrédients ok)"))
                 .otherwise(F.lit("Incomplète (Manque des données critiques)")))
    .groupBy("statut_recette")
    .count()
    # Ajout du pourcentage
    .withColumn("pourcentage", F.round((F.col("count") / total_rows_main) * 100, 2))
)

display(df_completude.toPandas())

--- Option 3 : Comparaison des recettes 'Parfaites', 'Partielles' et 'Incomplètes' ---


,statut_recette,count,pourcentage
0,Incomplète (Manque des données critiques),685178,66.54
1,Partielle (Image + Nutrition de base + Ingrédi...,291437,28.30
2,100% Parfaite (Toutes les colonnes des tables ...,53105,5.16


In [8]:
missing_stats = df_main.select([
    F.round((F.count(F.col(c)) / total_rows_main) * 100, 2).alias(c)
    for c in df_main.columns
])

# Astuce Jupyter : Pour créer un graphique équivalent à Databricks,
# on convertit en Pandas et on utilise la fonction de plot native.
pdf_missing = missing_stats.toPandas()
display(pdf_missing)
# Décommentez la ligne ci-dessous pour afficher un graphique en barres :
# pdf_missing.T.plot(kind='bar', title='Pourcentage de valeurs non-nulles', legend=False)

,recipe_id,title,description,instructions_text,ingredients_raw,ingredients_validated,n_ingredients_validated,n_steps,cook_minutes,cook_time_category,image_url,image_urls,has_image,source_url,energy_kcal,nutri_score,tags
0,100.0,100.0,29.65,100.0,100.0,100.0,100.0,30.54,30.54,100.0,98.76,99.49,100.0,100.0,34.24,100.0,30.54


## 5. Distributions Métiers & Statistiques
#### Analyse des répartitions sur des axes métiers clés pour comprendre la morphologie de notre catalogue de recettes.

In [9]:
# Distribution du Nutri-Score calculé
nutri_distribution = (df_main
    .filter(F.col("nutri_score").isNotNull())
    .groupBy("nutri_score")
    .count()
    .withColumn("percentage", F.round((F.col("count") / total_rows_main) * 100, 2))
    .orderBy("nutri_score")
)

display(nutri_distribution.toPandas())

,nutri_score,count,percentage
0,A,34111,3.31
1,B,52535,5.10
2,C,71715,6.96
3,D,72868,7.08
4,E,798491,77.54


In [10]:
# Ingrédients les plus célèbres
top_ingredients = (df_index
    .groupBy("ingredient")
    .count()
    .orderBy(F.col("count").desc())
    .limit(30)
)

display(top_ingredients.toPandas())

,ingredient,count
0,salt,362348
1,butter,237604
2,sugar,217878
3,olive oil,169619
4,water,158509
5,eggs,151156
6,garlic cloves,133413
7,milk,105037
8,flour,102239
9,onion,101551


In [11]:
# Analyse des temps de cuisson
cook_time_analysis = (df_main
    .filter(F.col("cook_time_category") != "inconnu")
    .groupBy("cook_time_category")
    .agg(
        F.count("recipe_id").alias("nombre_recettes"),
        F.round(F.avg("energy_kcal"), 1).alias("moyenne_calories"),
    )
    # Tri personnalisé pour l'affichage
    .orderBy(F.when(F.col("cook_time_category") == "rapide", 1)
              .when(F.col("cook_time_category") == "moyen", 2)
              .otherwise(3))
)

display(cook_time_analysis.toPandas())

,cook_time_category,nombre_recettes,moyenne_calories
0,rapide,128763,356.9
1,moyen,97657,448.0
2,long,88090,577.3


## 6. Valeurs Aberrantes (Outliers) & Profils Nutritionnels
#### Identification des recettes présentant des données irréalistes (erreurs de parsing de l'open data, fautes de frappe sur les temps de cuisson, etc.).

In [12]:
# Petit aperçu rapide de la nouvelle table des macros nutritionnelles
display(df_nutrition_detail.summary("count", "mean", "min", "max").toPandas())

outliers = (df_main
    .select("title", "n_ingredients_validated", "energy_kcal", "cook_minutes", "source_url")
    .filter(
        (F.col("n_ingredients_validated") > 40) |  # Plus de 40 ingrédients
        (F.col("energy_kcal") > 3000) |            # Plus de 3000 kcal
        (F.col("cook_minutes") > 1440)             # Cuisson > 24h
    )
    .orderBy(F.col("energy_kcal").desc_nulls_last())
    .limit(50)
)

display(outliers.toPandas())

,summary,recipe_id,fat_g,protein_g,salt_g,saturates_g,sugars_g
0,count,1029720,96331,96331,96331,96331,96331
1,mean,Infinity,15.963285374168098,5.407726978042251,2.1473195383694006,5.926701651785195,15.435612804464133
2,min,000018c8a5,0.0,0.0,0.0,0.0,0.0
3,max,ffffdea29a,100.0,77.96188,96.895,91.3,99.8


,title,n_ingredients_validated,energy_kcal,cook_minutes,source_url
0,Tennessee Moonshine,5,434360.187500,20,http://www.food.com/recipe/tennessee-moonshine...
1,Deep Fried Prime Rib,6,101614.703125,100,http://www.food.com/recipe/deep-fried-prime-ri...
2,Powdered Hot Cocoa Mix,4,45609.000000,10,http://www.food.com/recipe/powdered-hot-cocoa-...
3,MMMMMMMMilky Way Cake,9,44239.800781,60,http://www.food.com/recipe/mmmmmmmmilky-way-ca...
4,Caledonian Wedding Cake,17,43924.601562,630,http://www.food.com/recipe/caledonian-wedding-...
5,Seasoned Goldfish Crackers,7,30933.400391,30,http://www.food.com/recipe/seasoned-goldfish-c...
6,Ultimate Coconut Cake II,23,28930.199219,120,http://www.food.com/recipe/ultimate-coconut-ca...
7,Sinfully delicious Hot Chocolate Mix,4,25712.599609,5,http://www.food.com/recipe/sinfully-delicious-...
8,Luscious Coconut Cake,27,21190.099609,65,http://www.food.com/recipe/luscious-coconut-ca...
9,Mexican Ojarascas Cookies,11,19380.599609,40,http://www.food.com/recipe/mexican-ojarascas-c...


## 7. Vue Master (Dénormalisation)
#### Reconstruction d'une vue aplatie (`df_master`) croisant les métadonnées de base et les macros nutritionnels détaillés. Idéal pour des exports CSV ponctuels ou pour alimenter des dashboards de BI.

In [13]:
# Vue "Master" : Jointure des tables
df_master = df_main.join(df_nutrition_detail, on="recipe_id", how="left")

print(f"Nombre de colonnes dans la vue Master : {len(df_master.columns)}")

# Affichage des 10 premières lignes
display(df_master.limit(10).toPandas())

Nombre de colonnes dans la vue Master : 22


,recipe_id,title,description,instructions_text,ingredients_raw,ingredients_validated,n_ingredients_validated,n_steps,cook_minutes,cook_time_category,...,has_image,source_url,energy_kcal,nutri_score,tags,fat_g,protein_g,salt_g,saturates_g,sugars_g
0,5a616ae73b,Italian Wine Biscuits,another traditional italian cookie well known ...,Preheat oven to 375. | Mix wet ingredients int...,"[4 cups all-purpose flour, 1 cup sugar, 1 cup ...","[all - purpose flour, sugar, red table wine, b...",5,7.0,30.0,rapide,...,True,http://www.food.com/recipe/italian-wine-biscui...,177.000000,C,"[30-minutes-or-less, time-to-make, course, cui...",NaN,NaN,NaN,NaN,NaN
1,33b10f84ac,Pumpkin Creme Caramel,this is a delightful autumn dessert that is re...,Preheat oven to 350F. | Lightly coat 8 6-oz. |...,"[23 cup plus 1/2 cup sugar, divided, 1/2 tsp. ...","[sugar, salt, eggs, pumpkin puree, vanilla ext...",8,14.0,65.0,long,...,True,http://www.vegetariantimes.com/recipe/pumpkin-...,247.100006,C,"[time-to-make, course, main-ingredient, prepar...",NaN,NaN,NaN,NaN,NaN
2,f9c7945fd1,Beetroot Leaf and Ricotta Salad,this is suggested as an accompaniment to honey...,Place all items in a bowl and toss to combine.,"[150 g beet leaves (5 1/4 oz), 120 g ricotta c...","[beet leaves, ricotta cheese, olive oil, balsa...",4,1.0,6.0,rapide,...,True,http://www.food.com/recipe/beetroot-leaf-and-r...,167.827469,C,"[15-minutes-or-less, time-to-make, course, pre...",13.684026,5.114058,0.177157,4.384185,4.106709
3,68b2035a2c,Mini Parmesan Scones,NaN,Preheat oven to 350 degrees F (175 degrees C)....,"[2 tablespoons butter, 2 cups self-rising flou...","[butter, self - rising flour, salt, parmesan c...",7,NaN,NaN,inconnu,...,True,http://allrecipes.com/recipe/mini-parmesan-sco...,241.493927,C,None,7.627635,9.424111,1.899390,4.598870,1.272811
4,d9d9a6a6e5,Apple Spice Cake,a really easy cake that is absolutely delicious.,1. | Preheat oven to 350 degrees F. Grease pan...,"[1 (18 ounce) box spice cake mix, 1 (21 ounce)...","[spice cake mix, apple pie filling, eggs, crea...",7,4.0,35.0,moyen,...,True,http://www.food.com/recipe/apple-spice-cake-39...,229.000000,C,"[60-minutes-or-less, time-to-make, course, mai...",NaN,NaN,NaN,NaN,NaN
5,0e38daa3a7,Chocolate Cake Donuts,this recipe came on the packaging of my new do...,"For the donuts, mix dry ingredients in bowl. |...","[2 cups Flour, 3/4 cups Sugar, 1/2 cups Unswee...","[flour, sugar, unsweetened cocoa powder, bakin...",18,13.0,26.0,rapide,...,True,http://tastykitchen.com/recipes/breakfastbrunc...,181.199997,C,"[30-minutes-or-less, time-to-make, course, pre...",NaN,NaN,NaN,NaN,NaN
6,01283ea41b,Light and Easy Chicken Chili,"a filling, delicious meal that is healthy and ...",Heat olive oil in large saucepan with medium h...,"[2 teaspoons olive oil, 1 medium onion, choppe...","[olive oil, onion, garlic, ground chicken, chi...",13,9.0,45.0,moyen,...,True,http://www.food.com/recipe/light-and-easy-chic...,247.500000,C,"[60-minutes-or-less, time-to-make, course, mai...",NaN,NaN,NaN,NaN,NaN
7,0d8c5f04d4,Potato casserole,this is from home cooking magazine. three kin...,Preheat oven to 350 Fahrenheit | Wash the pota...,"[4 lb potatoes, 1 cup plain yogurt, I used gre...","[potatoes, plain yogurt, evaporated milk, shre...",9,6.0,55.0,moyen,...,True,https://cookpad.com/us/recipes/356071-potato-c...,183.841705,C,"[60-minutes-or-less, time-to-make, course, mai...",10.317756,10.045829,32.324520,3.559546,9.466318
8,541b6987bf,Old Fashioned Raisin Pie,NaN,Divide the pastry almost in half and roll out ...,"[1 each pastry 2 crust pie, 3/4 cup brown suga...","[pastry dough, brown sugar, lemon juice, butte...",8,14.0,0.0,rapide,...,True,https://recipeland.com/recipe/v/old-fashioned-...,257.138916,C,"[15-minutes-or-less, time-to-make, course, mai...",9.673600,2.769692,0.041123,2.117886,10.569148
9,e8410d37e8,Knock You Nakeds,"a sweet, gooey, chocolatey type of brownie bar...",Preheat oven to 180c& grease and line a 8 x 13...,"[1 box chocolate cake mix, 1 cup choppe

In [14]:
spark.stop()